In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
df = pd.read_csv(r"C:\Users\USER\Documents\MY PORTFOLIO\Smart Travel Recommendation System\Travel Agency\cleaned_data.csv")

In [3]:
df.head()

,channel,trip_type,cabin,airline,origin,destination,pax_count,lead_time,season,passenger_type,...,Payment-dep,Travel-dep,booking_year,booking_day,travel_year,travel_day,payment_year,payment_day,dep_year,dep_day
0,Online,One-way,PremiumEconomy,WY,SHJ,DAC,1,3,Spring,ADT,...,1,0,2023,27,2023,2,2023,1,2023,2
1,Online,One-way,Economy,ME,KWI,BEY,2,29,Spring,ADT,...,28,0,2016,17,2016,17,2016,18,2016,17
2,Online,One-way,Economy,ME,KWI,BEY,2,29,Spring,ADT,...,28,0,2016,17,2016,17,2016,18,2016,17
3,WhatsApp,One-way,PremiumEconomy,UK,KWI,CMB,5,58,Summer,ADT,...,58,0,2023,17,2023,14,2023,17,2023,14
4,WhatsApp,One-way,PremiumEconomy,UK,KWI,CMB,5,58,Summer,ADT,...,58,0,2023,17,2023,14,2023,17,2023,14


In [4]:
df.columns

Index(['channel', 'trip_type', 'cabin', 'airline', 'origin', 'destination',
       'pax_count', 'lead_time', 'season', 'passenger_type', 'gender',
       'nationality', 'loyalty_member', 'payment_method', 'paid_usd',
       'dep_airport', 'arr_airport', 'dep_city', 'arr_city', 'flight_duration',
       'stop_type', 'Booking-Travel', 'Booking-Payment', 'Payment-Travel',
       'Booking-dep', 'Payment-dep', 'Travel-dep', 'booking_year',
       'booking_day', 'travel_year', 'travel_day', 'payment_year',
       'payment_day', 'dep_year', 'dep_day'],
      dtype='object')

In [5]:
dfg = df.groupby(["season", "dep_city"])["arr_city"].apply(list).reset_index()

In [6]:
df = dfg["arr_city"]

In [7]:
df

0      [Kuwait City, Kuwait City, Bangkok, Bangkok, B...
1      [Jeddah, Jeddah, Jeddah, Doha, Doha, Doha, Doh...
2      [Medina, Medina, Colombo, Colombo, Najaf, Naja...
3      [Perth, Perth, Casablanca, Casablanca, Jeddah,...
4      [Tel Aviv, Tel Aviv, Kuwait City, Kuwait City,...
                             ...                        
323    [Casablanca, Casablanca, Hanoi, Hanoi, Hanoi, ...
324    [Colombo, Chennai, Chennai, Chennai, Dhaka, Dh...
325    [Hanoi, Hanoi, Hanoi, Amsterdam, Amsterdam, Am...
326    [Kuwait City, Kuwait City, Addis Ababa, Addis ...
327    [Kuwait City, Kuwait City, Kuwait City, Kuwait...
Name: arr_city, Length: 328, dtype: object

In [8]:
te = TransactionEncoder()
df_te = te.fit(df).transform(df)
df_te

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False,  True, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False,  True, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]], shape=(328, 82))

In [9]:
df = pd.DataFrame(df_te, columns=te.columns_)

In [10]:
df.head()

,Abu Dhabi,Addis Ababa,Adelaide,Algiers,Amman,Amsterdam,Athens,Baghdad,Bangkok,Barcelona,...,Sydney,Taipei,Tbilisi,Tel Aviv,Tokyo,Toronto,Vienna,Washington DC,Yerevan,Zurich
0,False,False,False,False,False,False,True,False,True,True,...,False,False,False,True,False,True,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False,True,True,...,False,False,True,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,True,True,False,False,False,True,False,True


In [11]:
mdf = apriori(df, min_support=0.05, use_colnames=True)

In [12]:
rules = association_rules(mdf, metric='lift', min_threshold=1.0)

In [13]:
rl = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rl

,antecedents,consequents,support,confidence,lift
0,(Addis Ababa),(Abu Dhabi),0.064024,0.411765,1.116189
1,(Abu Dhabi),(Addis Ababa),0.064024,0.173554,1.116189
2,(Adelaide),(Abu Dhabi),0.073171,0.393443,1.066522
3,(Abu Dhabi),(Adelaide),0.073171,0.198347,1.066522
4,(Abu Dhabi),(Algiers),0.070122,0.190083,1.355372
...,...,...,...,...,...
8011,"(Muscat, Kuwait City)","(Riyadh, Manama)",0.051829,0.161905,1.264399
8012,"(Manama, Kuwait City)","(Riyadh, Muscat)",0.051829,0.154545,1.152066
8013,(Riyadh),"(Muscat, Manama, Kuwait City)",0.051829,0.161905,1.295238
8014,(Muscat),"(Riyadh, Manama, Kuwait City)",0.051829,0.153153,1.357682


In [14]:
df = rl[rl["confidence"] > 0.69]

In [15]:
def clean_rules(rules):
    # Convert frozensets to clean strings
    rules = rules.copy()  # Avoid modifying original
    
    rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
    
    return rules


rl_clean = clean_rules(df)

rl = rl_clean.sort_values('confidence', ascending=False).reset_index(drop=True)
rl

,antecedents,consequents,support,confidence,lift
0,"Bengaluru, Yerevan",Dhaka,0.054878,0.900000,3.690000
1,"Barcelona, Hong Kong",Karachi,0.051829,0.894737,3.668421
2,"Colombo, Beirut",Dhaka,0.051829,0.894737,3.668421
3,"Bangkok, Manila",Dubai,0.057927,0.863636,2.575207
4,"Kolkata, Sialkot",Colombo,0.051829,0.850000,4.161194
...,...,...,...,...,...
132,"Mumbai, Peshawar",Dhaka,0.054878,0.692308,2.838462
133,"Dhaka, Peshawar",Mumbai,0.054878,0.692308,4.128671
134,"Mumbai, Peshawar",Islamabad,0.054878,0.692308,3.290970
135,"Karachi, Multan",Seoul,0.054878,0.692308,2.911243


In [16]:
rl[["antecedents", "consequents", "confidence"]].to_csv(r"C:\Users\USER\Documents\MY PORTFOLIO\Smart Travel Recommendation System\Travel Agency\Recommendation data.csv", index=False)